# **Backward Chaining Engine with Askable Facts**

In [ ]:
# Facts provided by the user
facts = {}

# Rules for diagnosing car faults
rules = {
    "dead_battery": {
        "conditions": {
            "lights_dim": True,
            "engine_cranking": False
        }
    },

    "fuel_problem": {
        "conditions": {
            "engine_cranking": True,
            "engine_starts": False
        }
    },

    # New starter motor fault
    "starter_motor_fault": {
        "conditions": {
            "clicking_sound": True,
            "engine_cranking": False
        }
    }
}


def ask_fact(fact):
    """Ask the user for a fact if it is not already known."""
    if fact not in facts:
        while True:
            answer = input(
                f"Is '{fact.replace('_', ' ')}' true? (yes/no): "
            ).strip().lower()

            if answer in ["yes", "y"]:
                facts[fact] = True
                break
            elif answer in ["no", "n"]:
                facts[fact] = False
                break
            else:
                print("Please enter yes or no.")

    return facts[fact]


def backward_chain(goal):
    """Check whether a goal can be proved using the rules."""
    if goal not in rules:
        return ask_fact(goal)

    for condition, expected_value in rules[goal]["conditions"].items():
        actual_value = ask_fact(condition)

        if actual_value != expected_value:
            return False

    return True


def diagnose():
    hypotheses = [
        "dead_battery",
        "fuel_problem",
        "starter_motor_fault"
    ]

    for hypothesis in hypotheses:
        print(f"\nChecking hypothesis: {hypothesis.replace('_', ' ')}")

        if backward_chain(hypothesis):
            print(f"\nDiagnosis: {hypothesis.replace('_', ' ')}")
            return hypothesis

    print("\nNo matching fault identified.")
    return None


diagnose()


Checking hypothesis: dead battery
Is 'lights dim' true? (yes/no): yes
Is 'engine cranking' true? (yes/no): no

Diagnosis: dead battery


'dead_battery'

In [ ]:
def diagnose_with_order(hypotheses):
    for hypothesis in hypotheses:
        print(
            f"\nChecking hypothesis: "
            f"{hypothesis.replace('_', ' ')}"
        )

        if backward_chain(hypothesis):
            print(
                f"\nDiagnosis: "
                f"{hypothesis.replace('_', ' ')}"
            )
            return hypothesis

    print("\nNo matching fault identified.")
    return None


# Change the order of hypotheses
facts.clear()

new_order = [
    "starter_motor_fault",
    "dead_battery",
    "fuel_problem"
]

diagnose_with_order(new_order)


Checking hypothesis: starter motor fault
Is 'clicking sound' true? (yes/no): yes
Is 'engine cranking' true? (yes/no): no

Diagnosis: starter motor fault


'starter_motor_fault'

In [ ]:
current_goal = None


def ask_fact(fact):
    """Ask for a fact and explain the question when WHY is entered."""
    if fact in facts:
        return facts[fact]

    while True:
        answer = input(
            f"Is '{fact.replace('_', ' ')}' true? "
            "(yes/no/why): "
        ).strip().lower()

        if answer == "why":
            print(
                f"WHY: The system is checking "
                f"'{current_goal.replace('_', ' ')}'."
            )
            print(
                f"Reason: The rule for "
                f"'{current_goal.replace('_', ' ')}' "
                f"requires '{fact}' to be "
                f"{'true' if rules[current_goal]['conditions'][fact] else 'false'}."
            )
            continue

        if answer in ["yes", "y"]:
            facts[fact] = True
            return True

        if answer in ["no", "n"]:
            facts[fact] = False
            return False

        print("Please enter yes, no, or why.")


def backward_chain(goal):
    global current_goal
    current_goal = goal

    if goal not in rules:
        return ask_fact(goal)

    for condition, expected_value in rules[goal]["conditions"].items():
        actual_value = ask_fact(condition)

        if actual_value != expected_value:
            return False

    return True


# Test the updated engine
facts.clear()
diagnose_with_order([
    "starter_motor_fault",
    "dead_battery",
    "fuel_problem"
])


Checking hypothesis: starter motor fault
Is 'clicking sound' true? (yes/no/why): yes
Is 'engine cranking' true? (yes/no/why): why
WHY: The system is checking 'starter motor fault'.
Reason: The rule for 'starter motor fault' requires 'engine_cranking' to be false.
Is 'engine cranking' true? (yes/no/why): yes

Checking hypothesis: dead battery
Is 'lights dim' true? (yes/no/why): why
WHY: The system is checking 'dead battery'.
Reason: The rule for 'dead battery' requires 'lights_dim' to be true.
Is 'lights dim' true? (yes/no/why): no

Checking hypothesis: fuel problem
Is 'engine starts' true? (yes/no/why): why
WHY: The system is checking 'fuel problem'.
Reason: The rule for 'fuel problem' requires 'engine_starts' to be false.
Is 'engine starts' true? (yes/no/why): no

Diagnosis: fuel problem


'fuel_problem'